# Examen Teórico

## Ricardo Calvo - A01028889

## Introducción

Este proyecto tiene como objetivo implementar un clasificador de texto utilizando el dataset 20 Newsgroups de Scikit-learn, el cual contiene miles de datos distribuidos en 20 categorías categorías distintas. Usando una red neuronal se busca entrenar un modelo capaz de identificar a qué categoría pertenece cada dato con la clasificación multiclase. Desupués del entrenamiento del modelo, evaluaremos con métricas de desempeño y la visualización de resultadosa a partir de gráficas, con el fin de analizar la eficacia del algiritmo propuesto en un ejemplo práctico de aprendizaje automático.

In [1]:
# Step 1
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
# Step 2
import plotly.express as px
import pandas as pd
# Step 3
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


### 1. Preprocess text 

In [2]:
# Datasets
train = fetch_20newsgroups(subset='train')  # Training Dataset
test = fetch_20newsgroups(subset='test')  # Test Dataset

# Vectorizar
max_features = 50000
num_classes = len(train.target_names)
vectorizer = TfidfVectorizer(
    stop_words='english', max_features=max_features, token_pattern=r"(?u)\b[a-zA-Z]{2,}\b")

# Input Vectors
train_X = vectorizer.fit_transform(train.data)
test_X = vectorizer.transform(test.data)

# Etiquetas
train_y = train.target
test_y = test.target

### 2 Visualize data distribution

In [3]:
class_names = train.target_names
train_df = pd.DataFrame({
    "Categoria": [class_names[i] for i in train_y]
})

fig = px.histogram(train_df, x="Categoria", title="Distribución de categorías en Train")
fig.update_xaxes(tickangle=45)
fig.show()


In [4]:
train_lengths = [len(text.split()) for text in train.data]
len_df = pd.DataFrame({"Longitud": train_lengths})

fig = px.box(len_df, y="Longitud", title="Distribución de longitudes de documentos (Train)")
fig.show()


### 3. Neuronal Network implementation

In [5]:
k_train_X = tf.convert_to_tensor(train_X.toarray(), dtype=tf.float32)
k_test_X = tf.convert_to_tensor(test_X.toarray(), dtype=tf.float32)

model = models.Sequential([
    layers.Input(shape=(max_features,)),
    layers.Dense(256, activation='relu',
                 kernel_regularizer=tf.keras.regularizers.l2(5e-4)),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu',
                 kernel_regularizer=tf.keras.regularizers.l2(5e-4)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax'),
])


### 4. Train and adjust model

In [6]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=6e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
]

history = model.fit(k_train_X, train_y,
                    validation_data=(k_test_X, test_y),
                    batch_size=256, epochs=20,
                    callbacks=callbacks)

Epoch 1/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 8s 161ms/step - accuracy: 0.3444 - loss: 3.0461 - val_accuracy: 0.6037 - val_loss: 2.8525 - learning_rate: 6.0000e-04
Epoch 2/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 156ms/step - accuracy: 0.6277 - loss: 2.4592 - val_accuracy: 0.7740 - val_loss: 2.1382 - learning_rate: 6.0000e-04
Epoch 3/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 156ms/step - accuracy: 0.7502 - loss: 1.6951 - val_accuracy: 0.8113 - val_loss: 1.6293 - learning_rate: 6.0000e-04
Epoch 4/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 157ms/step - accuracy: 0.8467 - loss: 1.2904 - val_accuracy: 0.8224 - val_loss: 1.4210 - learning_rate: 6.0000e-04
Epoch 5/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 157ms/step - accuracy: 0.9033 - loss: 1.0856 - val_accuracy: 0.8374 - val_loss: 1.3126 - learning_rate: 6.0000e-04
Epoch 6/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 156ms/step - accuracy: 0.9387 - loss: 0.9519 - val_accuracy: 0.8408 - val_loss: 1.2512 - learning_rate: 6.0000e-04
Epoch 7/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 155ms/step - accuracy: 

### 5. Visualize learning curve

In [8]:
df = pd.DataFrame(history.history)
df["epoch"] = range(1, len(df)+1)

fig_loss = px.line(df, x="epoch", y=[c for c in df.columns if c in ["loss", "val_loss"]],
                   title="Curvas de pérdida")
fig_loss.update_layout(xaxis_title="Época", yaxis_title="Loss")
fig_loss.show()

# Curvas de accuracy
fig_acc = px.line(df, x="epoch", y=[c for c in df.columns if c in ["accuracy", "val_accuracy"]],
                  title="Curvas de accuracy")
fig_acc.update_layout(xaxis_title="Época", yaxis_title="Accuracy")
fig_acc.show()


### 6. Evaluate performance using performance metrics

### 7. Confession matrix 

### 8. Experiment with different network architectures

### 9. Test with k-fold cross validation

### 10. ROC & AUC

### 11. Findings